# 🧠 Red Neuronal - Trabajo Final de Grado
## Predicción del Riesgo País Latinoamericano

**Autor:** Martín | **Universidad:** UNC

---
Secciones: Config → Datos → Modelo → Entrenamiento → Predicción → Comparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
import warnings, os, sys

sys.path.append(os.path.abspath('..'))
from src.utils import metricas_prediccion
warnings.filterwarnings('ignore')

# ELEGIR FRAMEWORK (descomentar uno)
# import tensorflow as tf; from tensorflow import keras
# from keras.models import Sequential
# from keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
# from keras.callbacks import EarlyStopping, ReduceLROnPlateau
# FRAMEWORK = 'tensorflow'

# import torch; import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset
# FRAMEWORK = 'pytorch'

FRAMEWORK = None
print('Descomentar framework arriba ⬆️')

In [ ]:
# CARGAR DATOS
DATA_PATH = os.path.join('..', 'data', 'processed', 'series_consolidadas.csv')
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True) if os.path.exists(DATA_PATH) else pd.DataFrame()
print(f'Datos: {df.shape}') if not df.empty else print('❌ Ejecutar 01_base_de_datos.ipynb primero')

In [ ]:
# CONFIGURACIÓN
TARGET = 'EMBI+ Argentina'
FEATURES = []  # Descomentar variables a usar
# FEATURES = ['VIX', 'US Treasury 10Y', ...]
if not FEATURES: FEATURES = [c for c in df.columns if c != TARGET]

LOOKBACK = 30; HORIZON = 1
TRAIN_RATIO = 0.70; VAL_RATIO = 0.15
BATCH_SIZE = 32; EPOCHS = 100; LR = 0.001
HIDDEN = 64; N_LAYERS = 2; DROPOUT = 0.2
ARCH = 'LSTM'  # 'LSTM', 'GRU', 'BiLSTM'
print(f'Target: {TARGET} | Features: {len(FEATURES)} | Arch: {ARCH}')

In [ ]:
# CREAR SECUENCIAS
def crear_secuencias(data, target_col, lookback, horizon=1):
    X, y = [], []
    idx = data.columns.get_loc(target_col)
    for i in range(lookback, len(data) - horizon + 1):
        X.append(data.iloc[i-lookback:i].values)
        y.append(data.iloc[i+horizon-1, idx])
    return np.array(X), np.array(y)

cols = [TARGET] + FEATURES
df_nn = df[cols].dropna()
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_nn), columns=df_nn.columns, index=df_nn.index)
target_scaler = MinMaxScaler()
target_scaler.fit(df_nn[[TARGET]])

X, y = crear_secuencias(df_scaled, TARGET, LOOKBACK, HORIZON)
n = len(X); n_train = int(n*TRAIN_RATIO); n_val = int(n*(TRAIN_RATIO+VAL_RATIO))
X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_val], y[n_train:n_val]
X_test, y_test = X[n_val:], y[n_val:]
print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

## Modelo TensorFlow/Keras

In [ ]:
if FRAMEWORK == 'tensorflow':
    model = Sequential()
    n_feat = X_train.shape[2]
    RNNLayer = LSTM if ARCH != 'GRU' else GRU
    for i in range(N_LAYERS):
        ret = i < N_LAYERS - 1
        layer = RNNLayer(HIDDEN, return_sequences=ret)
        if i == 0:
            if ARCH == 'BiLSTM': model.add(Bidirectional(LSTM(HIDDEN, return_sequences=ret), input_shape=(LOOKBACK, n_feat)))
            else: model.add(RNNLayer(HIDDEN, return_sequences=ret, input_shape=(LOOKBACK, n_feat)))
        else:
            model.add(layer)
        model.add(Dropout(DROPOUT))
    model.add(Dense(32, activation='relu')); model.add(Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss='mse', metrics=['mae'])
    model.summary()

## Modelo PyTorch

In [ ]:
if FRAMEWORK == 'pytorch':
    class RiskRNN(nn.Module):
        def __init__(self, inp, hid, layers, drop, arch):
            super().__init__()
            RNN = nn.LSTM if arch == 'LSTM' else nn.GRU
            self.rnn = RNN(inp, hid, layers, batch_first=True, dropout=drop)
            self.fc = nn.Sequential(nn.Linear(hid, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))
        def forward(self, x):
            out, _ = self.rnn(x)
            return self.fc(out[:, -1, :]).squeeze()
    model = RiskRNN(X_train.shape[2], HIDDEN, N_LAYERS, DROPOUT, ARCH)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
    print(model)

## Entrenamiento

In [ ]:
if FRAMEWORK == 'tensorflow':
    cb = [EarlyStopping('val_loss', patience=15, restore_best_weights=True),
          ReduceLROnPlateau('val_loss', factor=0.5, patience=5, min_lr=1e-6)]
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cb, verbose=1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=history.history['loss'], name='Train'))
    fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Val'))
    fig.update_layout(title='Loss', template='plotly_dark', height=350)
    fig.show()
elif FRAMEWORK == 'pytorch':
    best_vl = float('inf'); patience = 0
    X_vt, y_vt = torch.FloatTensor(X_val), torch.FloatTensor(y_val)
    for ep in range(EPOCHS):
        model.train(); el = 0
        for bx, by in train_loader:
            optimizer.zero_grad(); p = model(bx); l = criterion(p, by); l.backward(); optimizer.step(); el += l.item()
        model.eval()
        with torch.no_grad(): vl = criterion(model(X_vt), y_vt).item()
        if vl < best_vl: best_vl = vl; patience = 0; bs = model.state_dict().copy()
        else:
            patience += 1
            if patience >= 15: print(f'Early stop ep {ep+1}'); break
        if (ep+1) % 10 == 0: print(f'Ep {ep+1} | Train: {el/len(train_loader):.6f} | Val: {vl:.6f}')
    model.load_state_dict(bs)

## Predicción y Evaluación

In [ ]:
if FRAMEWORK == 'tensorflow': yp = model.predict(X_test).flatten()
elif FRAMEWORK == 'pytorch':
    model.eval()
    with torch.no_grad(): yp = model(torch.FloatTensor(X_test)).numpy()
else: yp = np.array([])

if len(yp):
    yp_r = target_scaler.inverse_transform(yp.reshape(-1,1)).flatten()
    yt_r = target_scaler.inverse_transform(y_test.reshape(-1,1)).flatten()
    m = metricas_prediccion(yt_r, yp_r)
    print('📊 Métricas Test:'); [print(f'  {k}: {v}') for k,v in m.items()]
    dates = df_nn.index[-len(yt_r):]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dates, y=yt_r, name='Real'))
    fig.add_trace(go.Scatter(x=dates, y=yp_r, name='Predicción', line=dict(dash='dash')))
    fig.update_layout(title=f'{ARCH} - Predicción Riesgo País', template='plotly_dark', height=450)
    fig.show()

## Comparación con Modelos Econométricos
Completar con métricas del notebook 02

In [ ]:
comp = pd.DataFrame({'Modelo': ['LSTM','GRU','OLS','VAR','ARDL'],
    'RMSE': [np.nan]*5, 'MAE': [np.nan]*5, 'R²': [np.nan]*5})
print('⚠️ Completar con métricas del notebook 02')
display(comp.set_index('Modelo'))

## Extensión: Riesgo País LATAM
Países: Argentina, Brasil, México, Colombia, Chile, Perú

TODO: Cargar EMBI+ de cada país y entrenar modelos